<img src="logo.png" alt="Vegeta" width="240">

# Kestrel-4Q-2 against 4Q-1 — the back transition's remedies, the flying machine, and the case for 4Q-3

Notebook 40 left two aircraft. **Kestrel-4Q-1**: round struts, the X, elevons as effectors — it flies every transition
from the cruise speed and spends 110 W doing 16 m/s. **Kestrel-4Q-2**: faired posts, no diagonals, the reflexed
section — 59 W at 16 m/s, twice the endurance, and a back transition that balloons 12–16 m from the cruise speed,
because a clean aircraft cannot throw its speed away. The question this notebook answers: **what gets 4Q-2 from
horizontal to vertical without the rule "slow to 13 m/s first"** — and, with the two aircraft side by side as
flying machines (battery life, speed, climb, dive, the turn, the pursuit), what the third aircraft should be.

A kestrel hunts by hovering and stooping: the name asks for a quick climb, a controlled dive, a tight turn and the
patience of a long loiter. The tables at the end are read that way.

The aircraft are rebuilt from notebook 40's recipe (`kestrel2.vehicles`: the same CAD parameters, mass tables, lattice,
drag build-ups, Zephyr section numbers and elevon τ), and 38's own transition figures of 4Q-1 are shown beside 4Q-2's.

In [ ]:
import json, math, os, shutil, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from vegeta.labkit import bootstrap; bootstrap.lab("aero", False)  # the lab's folders on sys.path
import kestrel, kestrel_systems as ks, kestrel_flight as kf, kestrel_control as kc, kestrel_transition as kt, kestrel_drawings as kd
import kestrel2 as k2, kestrel_loop as kl, kestrel_report as kr

pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 230); pd.set_option("display.max_columns", 30)
OUT = Path("output/42_kestrel2"); OUT.mkdir(parents=True, exist_ok=True)
FIG38 = Path("designs/data/figures")
QUICK = os.environ.get("KESTREL_QUICK") == "1"
DESIGN38 = json.loads(Path("designs/data/kestrel_design.json").read_text()) if Path("designs/data/kestrel_design.json").exists() else {}
REP38 = kr.load_json("40_kestrel")
print("notebook 40's record:", "found" if DESIGN38 else "missing", "| its JSON report:", "found" if REP38 else "missing")

## 1. What notebook 40 calculated

The two aircraft as 38 left them — its iteration log is the shortest statement of the path from the envelope to 4Q-2 —
and the vehicles rebuilt here from the same recipe (the masses and Cd0 must match 38's record).

In [ ]:
if REP38:
    it = REP38["sections"].get("the iteration log")
    if it:
        display(pd.DataFrame(it["rows"]).set_index("_index")[[c for c in ("what changed", "take-off mass [kg]", "static margin [chords]", "stall speed [m/s]", "Cd0", "L/D at the cruise", "cruise power [W]", "endurance at the cruise [min]", "transitions passing", "checks holding") if c in it["columns"]]])
    opt = REP38["sections"].get("Kestrel-4Q-2 against 4Q")
    if opt:
        display(pd.DataFrame(opt["rows"]).set_index("_index"))
VEH = k2.vehicles(which=("4Q-1", "4Q-2"))
V1, V2 = VEH["4Q-1"], VEH["4Q-2"]
SEC = k2.section_numbers()
display(pd.DataFrame({n: {"mass [kg]": v.mass, "CG x [mm]": v.x_cg * 1000, "Cd0": v.pol.cd0, "CL max": v.pol.cl_max, "section": v.pol.source, "elevon τ": v.elevon_tau,
                          "elevons from [mm]": (v.elevon_y_in or 0) * 1000, "stall speed [m/s]": k2.stall_speed(v)} for n, v in VEH.items()}).T)
if DESIGN38:
    print("38's record: 4Q mass", DESIGN38.get("mass", {}).get("mass_kg"), "| Cd0 round / faired", DESIGN38.get("Cd0"))

## 2. The transitions, side by side

Left: notebook 40's figures of 4Q-1 (exported by 38, `designs/data/figures/`). Right: 4Q-2 flown here through the same
3 s schedules from the cruise speed. The path figure puts both on one axis.

In [ ]:
NOM = {n: {k: kt.simulate(v, k, t_tr=3.0) for k in ("forward", "back")} for n, v in VEH.items()}
for k in ("forward", "back"):
    png = FIG38 / f"40_kestrel_4q1_{k}_3s.png"
    if png.exists():
        print(f"notebook 40 — Kestrel-4Q-1, {k}, 3 s:"); display(Image(filename=str(png), width=1100))
    kd.episode_figure(NOM["4Q-2"][k], V2); plt.gcf().savefig(OUT / f"4q2_{k}_3s.png", dpi=110, bbox_inches="tight"); plt.show()
fig, axs = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for ax, k in zip(axs, ("forward", "back")):
    for n, v in VEH.items():
        g = NOM[n][k].log
        ax.plot(g["X"], g["Z"], label=f"{v.name}: gain {NOM[n][k].metrics['height gain [m]']:.1f} m, loss {NOM[n][k].metrics['height loss [m]']:.1f} m, pass {NOM[n][k].metrics['pass']}")
    ax.axhline(30, color="k", lw=0.6, ls=":"); ax.set(ylabel="height [m]", title=f"{k} transition, 3 s schedule, still air, entered at the cruise speed"); ax.legend(fontsize=8)
axs[1].set_xlabel("x [m]"); plt.tight_layout(); plt.savefig(OUT / "paths_4q1_4q2.png", dpi=110); plt.show()
NOMT = pd.DataFrame({f"{n} {k}": NOM[n][k].metrics for n in VEH for k in ("forward", "back")}).T[k2.COLS]
display(NOMT.astype({c: float for c in k2.COLS if c != "pass"}).round(2))

## 3. The back transition's remedies for 4Q-2

Every remedy through the matrix — 1.5, 3 and 5 s schedules, still air and a 6 m/s headwind — against the criteria of
notebook 40 (height within 3 m below and 10 m above the start, pitch within 10°, roll demand within the authority):

| remedy | what it is in the model |
|---|---|
| slow entry at 13 m/s | the rule of notebook 40: decelerate in level flight first, then pitch up |
| crow-brake elevons | through the schedule the upper wing's elevons go up and the lower's down: the pitch cancels, the drag of all four stays (flat-plate on the deflected area); the pitch travel left is ±25° − the brake |
| pitch past the vertical | the pitch reference bulges 20° beyond 90° at the schedule's middle: the thrust vector tilted aft brakes |
| reverse thrust | bidirectional drives: through the schedule the least thrust is −3 N per rotor instead of the idle, and the height loop may brake and pull down |
| descend into it | the height reference descends at 2 m/s from the start to the schedule's middle, so that the balloon lands back on the datum (the height criteria read against the start height) |
| crow brake + reverse | both |

Then the gusts (±3 m/s vertical, at the schedule's middle) for the remedies that pass, and the level deceleration
each option gives before the pitch-up (time, distance, energy from 16 to 13 m/s).

In [ ]:
durs = (3.0,) if QUICK else (1.5, 3.0, 5.0)
REM, EPS = k2.remedy_matrix(V2, durations=durs, headwinds=(0.0, 6.0), progress=True)
display(REM[k2.COLS].astype({c: float for c in k2.COLS if c != "pass"}).round(2))
RV = k2.remedy_verdict(REM)
display(RV.round(2))
REM.to_csv(OUT / "remedies_4q2.csv")
GUSTS = {}
for r, kw in k2.REMEDIES.items():
    if RV.loc[r, "passes"].startswith(str(len(durs) * 2)):
        GUSTS[r] = {f"gust {g:+g} m/s": kt.simulate(V2, "back", t_tr=3.0, gust_up=g, **kw).metrics["pass"] for g in (3.0, -3.0)}
GT = pd.DataFrame(GUSTS).T if GUSTS else pd.DataFrame()
print("gusts, 3 s, the remedies that pass the matrix:"); display(GT)
DEC = pd.DataFrame({"rotors at idle": k2.decelerate_level(V2), "crow brake ±20°": k2.decelerate_level(V2, airbrake_deg=20.0), "reverse thrust 3 N": k2.decelerate_level(V2, reverse_N=3.0),
                    "brake + reverse": k2.decelerate_level(V2, airbrake_deg=20.0, reverse_N=3.0), "4Q-1 at idle (for the contrast)": k2.decelerate_level(V1)}).T
display(DEC.round(2))
# the best remedies flown: the episode figures and the paths against the as-is case
fig, ax = plt.subplots(figsize=(14, 4.5))
for r in k2.REMEDIES:
    ep = EPS[(r, 3.0, 0.0)]
    ax.plot(ep.log["X"], ep.log["Z"], label=f"{r}: gain {ep.metrics['height gain [m]']:.1f} m, pass {ep.metrics['pass']}")
ax.axhline(30, color="k", lw=0.6, ls=":"); ax.axhline(40, color="r", lw=0.6, ls=":"); ax.set(xlabel="x [m]", ylabel="height [m]", title="Kestrel-4Q-2, back transition, 3 s, still air: the remedies"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "remedies_paths.png", dpi=110); plt.show()
for r in ("reverse thrust 3 N per rotor", "crow brake + reverse thrust"):
    kd.episode_figure(EPS[(r, 3.0, 0.0)], V2, title=f"Kestrel-4Q-2, back, 3 s: {r}"); plt.show()

## 4. The two aircraft as flying machines

Battery life and range against speed, the thrust-limited maximum speed, the climb (specific excess power P_s = (T − D) V / W:
the climb rate at each speed; the time to 100 m at the best rate; the vertical climb's margin from the hover), the
dive (the steady speed at idle thrust at 30°, 60° and straight down — what the airframe reaches, not what it survives:
the never-exceed speed is a structural number this study has not set), the level acceleration, the turn at the
corner speed (where the lift limit meets the 4 g structural limit) and the sustained turn at 16 m/s, and the
manoeuvrability in the horizontal plane from notebook 40's model (pitch, roll, yaw at 16 m/s).

In [ ]:
PERF = {n: k2.performance(v) for n, v in VEH.items()}
PUR = k2.pursuit(PERF)
display(PUR.astype(float).round(2))
fig, axs = plt.subplots(2, 2, figsize=(15, 9))
for n, pf in PERF.items():
    t = pf["table"]
    axs[0, 0].plot(t.index, t["endurance [min]"], label=n); axs[0, 1].plot(t.index, t["range [km]"], label=n)
    axs[1, 0].plot(t.index, t["P_s (climb rate) [m/s]"], label=n); axs[1, 1].plot(t.index, t["n available (lift)"], label=n)
axs[0, 0].set(xlabel="V [m/s]", ylabel="min", title="endurance in level flight"); axs[0, 1].set(xlabel="V [m/s]", ylabel="km", title="range")
axs[1, 0].set(xlabel="V [m/s]", ylabel="m/s", title="specific excess power: the climb rate at each speed"); axs[1, 0].axhline(0, color="k", lw=0.6)
axs[1, 1].set(xlabel="V [m/s]", ylabel="g", title="load factor the lift allows (capped at the 4 g structure)")
for a_ in axs.flat: a_.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "performance.png", dpi=110); plt.show()
for n, pf in PERF.items():
    print(n, "— manoeuvrability in the horizontal plane:"); display(pf["manoeuvrability"].round(2))

## 5. Kestrel-4Q-3

What the two aircraft proved: the clean airframe is the better flying machine on every count but one — it cannot
shed speed — and the one thing that closes that (reverse thrust on bidirectional drives, the crow brake with it)
costs no mass and a few hundred joules per transition. 4Q-3 is therefore 4Q-2's airframe with the brakes built in
and the elevons widened for them, flown through the same matrix and the same performance tables.

In [ ]:
PROP = k2.propose_4q3()
display(pd.Series(PROP, name="Kestrel-4Q-3").to_frame())
V3 = k2.vehicles(which=("4Q-3",))["4Q-3"]
R3 = {"as is (entered at 16 m/s)": {}, "reverse thrust 3 N per rotor": {"reverse_N": 3.0}, "crow brake ±20° + reverse thrust": {"airbrake_deg": 20.0, "reverse_N": 3.0}, "slow entry at 13 m/s (the fallback)": {"V_entry": 13.0}}
REM3, EPS3 = k2.remedy_matrix(V3, remedies=R3, durations=durs, headwinds=(0.0, 6.0), progress=True)
RV3 = k2.remedy_verdict(REM3)
display(RV3.round(2))
FWD3 = {tt: kt.simulate(V3, "forward", t_tr=tt).metrics["pass"] for tt in durs}
G3 = {f"back, gust {g:+g} m/s, brake + reverse": kt.simulate(V3, "back", t_tr=3.0, gust_up=g, airbrake_deg=20.0, reverse_N=3.0).metrics["pass"] for g in (3.0, -3.0)}
print("4Q-3 forward transitions:", FWD3, "| gusts:", G3)
PERF["4Q-3"] = k2.performance(V3)
PUR = k2.pursuit(PERF)
display(PUR.astype(float).round(2))
CHK = [kl.check("4Q-2 as is: back transitions from the cruise speed passing", float(RV.loc["as is (entered at 16 m/s)", "passes"].split(" of ")[0]) / (len(durs) * 2), "≥", 1.0),
       kl.check("4Q-2 with reverse thrust: passing", float(RV.loc["reverse thrust 3 N per rotor", "passes"].split(" of ")[0]) / (len(durs) * 2), "≥", 1.0),
       kl.check("4Q-2 with the crow brake alone: passing", float(RV.loc["crow-brake elevons ±20°", "passes"].split(" of ")[0]) / (len(durs) * 2), "≥", 1.0),
       kl.check("4Q-2 descending into it: passing", float(RV.loc["descend into it at 2 m/s", "passes"].split(" of ")[0]) / (len(durs) * 2), "≥", 1.0),
       kl.check("4Q-3 brake + reverse: back transitions passing", float(RV3.loc["crow brake ±20° + reverse thrust", "passes"].split(" of ")[0]) / (len(durs) * 2), "≥", 1.0),
       kl.check("4Q-3 forward transitions passing", sum(FWD3.values()) / len(FWD3), "≥", 1.0), kl.check("4Q-3 gusts passing", sum(G3.values()) / len(G3), "≥", 1.0),
       kl.check("4Q-3 endurance at 16 m/s / 4Q-1", PUR.loc["endurance at 16 m/s [min]", "4Q-3"] / PUR.loc["endurance at 16 m/s [min]", "4Q-1"], "≥", 1.5),
       kl.check("4Q-3 time for 180° at the corner / 4Q-1", PUR.loc["time for 180° at the corner [s]", "4Q-3"] / PUR.loc["time for 180° at the corner [s]", "4Q-1"], "≤", 1.0),
       kl.check("level deceleration 16 → 13 m/s with reverse [s]", float(DEC.loc["reverse thrust 3 N", "time [s]"]), "≤", 2.0)]
display(kl.check_table(CHK, "Kestrel-4Q-3"))

## 6. Findings

*(written after the run)*

## 7. The dossier

This notebook's numbers as JSON (`designs/data/reports/42_kestrel2.json`) and Markdown, stitched into `designs/data/kestrel_dossier.json` / `.md`.

In [ ]:
REPORT = {"the aircraft": pd.DataFrame({n: {"mass [kg]": v.mass, "Cd0": v.pol.cd0, "CL max": v.pol.cl_max, "section": v.pol.source, "elevon τ": v.elevon_tau, "stall speed [m/s]": k2.stall_speed(v)} for n, v in {**VEH, "4Q-3": V3}.items()}).T,
          "nominal transitions, 3 s": NOMT, "remedies, 4Q-2 (matrix)": REM[k2.COLS], "remedies, 4Q-2 (verdict)": RV, "gusts": GT, "level deceleration 16 → 13 m/s": DEC,
          "performance tables": {n: pf["table"].loc[[v for v in (10.0, 12.0, 14.0, 16.0, 18.0, 20.0, 25.0, 30.0) if v in pf["table"].index]] for n, pf in PERF.items()},
          "pursuit": PUR, "manoeuvrability": {n: pf["manoeuvrability"] for n, pf in PERF.items()}, "Kestrel-4Q-3": PROP, "4Q-3 remedies": RV3, "4Q-3 forward": FWD3, "4Q-3 gusts": G3,
          "checks": kl.check_table(CHK, "Kestrel-4Q-3")}
EXP = kr.export("42_kestrel2", "Kestrel-4Q-2 against 4Q-1 — notebook 42, the back transition's remedies and the case for 4Q-3", REPORT,
                meta={"4Q-1 mass [kg]": V1.mass, "4Q-2 mass [kg]": V2.mass, "section": SEC.get("pick")})
print("JSON report:", EXP["json"], f"({EXP['json_kB']} kB) | JSON dossier:", EXP["dossier_json"], f"({EXP['dossier_json_kB']} kB) | Markdown:", EXP["markdown"])
kr.paste("42_kestrel2", max_lines=30)